# Step-by-step demo of NEDAS using the vort2d model



## Install NEDAS and dependencies
NEDAS is available from the PyPI platform. You can install with ```pip install NEDAS```.

You can also clone the `NEDAS` repository and install in editable mode ```cd NEDAS; pip install -e .``` for active code development.

In [ ]:
# 1. Install the latest version of NEDAS on develop branch
%cd ~
%rm -rf NEDAS
!git clone https://github.com/nansencenter/NEDAS.git
%cd NEDAS
%pip install .

# 2. Install additional dependencies
# numba provides JIT compilation of python function to speed it up during runtime
%pip install numba
# cmocean provides better colormaps for visualization
%pip install cmocean

# 3. If in Google Colab, need to clone the tutorial repo too
import sys
if 'google.colab' in sys.modules:
    %mkdir ~/work
    %cd ~/work
    %rm -rf NEDAS_tutorials
    !git clone https://github.com/myying/NEDAS_tutorials.git

%cd ~/work/NEDAS_tutorials


In [10]:
# utility modules
import os
import numpy as np
from datetime import datetime, timedelta

# a few graphic modules
import matplotlib.pyplot as plt
import cmocean
from matplotlib import cm
from matplotlib.animation import FuncAnimation
from IPython.core.display import HTML


Check how many CPUs are available on your system. You can then set `nproc`, `nproc_util` in the configuration accordingly

**Note**: If `mpi4py` is not supported, you shall use `nproc=1`


In [ ]:
os.cpu_count()

## Configuration

A YAML file `vort2d/config.yml` contains all the settings for an experiment.

See [Configuration file](https://nedas.readthedocs.io/en/latest/config_file.html) documentation for more details.

In command line, you can run the experiment with `python -m NEDAS -c vort2d/config.yml`

Here we setup the `config` object in an interactive environment:

In [4]:
from NEDAS.config import Config

# load configuration YAML file
# additional kwargs will overwrite the values in the file
config = Config(config_file="vort2d/config.yml", nproc=1, quiet=False)

# you can also change values like this
config.debug = False
config.io_mode = 'offline'

# to list all configuration variables
# you can check config.__dict__

In [95]:
config.cycle_period = 1

config.model_def['vort2d']['dx'] = 9000

In [96]:
# Once you are happy with the configuration,
# you can initialize the main Scheme object
from NEDAS import get_scheme
scheme = get_scheme(config)

In [97]:
# For convenience, we make alias of a few frequently used objects

# model object, which is a Vort2DModel(Model) instance
model = scheme.c.models['vort2d']
from NEDAS.models.vort2d import Vort2DModel
assert isinstance(model, Vort2DModel)

# dataset object, which is a Vort2DObs(SyntheticObs) instance
dataset = scheme.c.datasets['vort2d']
from NEDAS.datasets.vort2d import Vort2DObs
assert isinstance(dataset, Vort2DObs)

# and the analysis grid (same as model grid), which is a RegularGrid instance
from NEDAS.grid import RegularGrid
grid = scheme.c.grid


## Prepare truth

We will run an Observing System Simulation Experiment (OSSE) for the vort2d model.

First step is to generate a "verifying truth", or a "nature run", which will be used both for generating synthetic observations and for verification of the results.

In [98]:
# set time to the start of the experiment
scheme.c.time = config.time_start

# clear previous results
%rm -rf vort2d/work/truth

# for the truth, we place the vortex in the center of the domain (without perturbing its position)
model.loc_sprd = 0

# run the model from time_start to time_end and save results
# in offline IO mode, the truth states are saved under model.truth_dir (vort2d/work/truth/*nc files)
scheme.run_step('prepare_truth')



Running prepare_truth step:                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
├── Generate vort2d truth ─────────────────────── ✅    15.65s                                                                                                                                          

In [99]:
scheme.c.time = config.time_start
truth_state = []
times = []
while scheme.c.time < config.time_end:
    times.append(scheme.c.time)

    truth = scheme.c.io.call_method(scheme.c, 'truth', model.read_var, name='velocity', member=None, time=scheme.c.time, model_src='vort2d')
    truth_state.append(truth)

    scheme.c.time = scheme.c.next_time

In [100]:
from NEDAS.utils.spatial_operation import gradx, grady

def uv2zeta(vel):
    u, v = vel[0], vel[1]
    zeta = gradx(v, grid.dx, grid.cyclic_dim) - grady(u, grid.dy, grid.cyclic_dim)
    return zeta

vort_cmap = getattr(cmocean.cm, 'curl')
vort_min = -5e-3
vort_max = 5e-3

In [114]:
def rmse(Xens, Xt):
    return np.sqrt(np.mean((np.mean(Xens, axis=0)-Xt)**2))

def sprd(Xens):
    return np.sqrt(np.mean(np.std(Xens, axis=0)**2))

def sawtooth(out_b, out_a):
    nt = out_b.size
    tt = np.zeros(nt*2)
    tt[0::2] = np.arange(0, nt)
    tt[1::2] = np.arange(0, nt)
    out_st = np.zeros(nt*2)
    out_st[0::2] = out_b
    out_st[1::2] = out_a
    return tt, out_st

In [113]:
from NEDAS.diag.metrics.spectral import pwrspec2d

def err_spec(Xens, Xt):
    wn, err_pwr = pwrspec2d(np.mean(Xens, axis=0)-Xt)
    return wn, err_pwr

def sprd_spec(Xens):
    ni, nj, nv, nens = Xens.shape
    Xmean = np.mean(Xens, axis=3)
    nup = int(np.ceil((ni+1)/2))
    pwr_ens = np.zeros((nup, nens))
    for m in range(nens):
        wn, pwr_ens[:, m] = pwr_spec(Xens[:, :, :, m] - Xmean)
    sprd_pwr = np.sum(pwr_ens, axis=1) / (nens-1)
    return wn, sprd_pwr

## Prepare initial ensemble

Second,

In [103]:
%rm -rf vort2d/work/init_ens

In [104]:
scheme.c.time = config.time_start
model.loc_sprd = 10000
scheme.run_step('prepare_init_ensemble')


Running prepare_init_ensemble step:                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            
├── Generate vort2d init ensemble ─────────────── ✅     0.85s (all 16 jobs done)                                                                                                                       

In [105]:
%rm -rf vort2d/work/cycle

In [ ]:
scheme.c.progress.call_stack = []
scheme.c.progress.push('')
scheme.c.time = config.time_start
scheme.c.log_event("CYCLING START...")
while scheme.c.time < config.time_end:
    scheme.c.log_event(f"CURRENT CYCLE: {scheme.c.time} -> {scheme.c.next_time}")
    scheme.run_step('preprocess')
    # DA step
    scheme.run_step('postprocess')
    scheme.run_step('ensemble_forecast')
    scheme.c.time = scheme.c.next_time
scheme.c.log_event("CYCLING COMPLETE.", flag='finish')

In [107]:
scheme.c.time = config.time_start
forecast_state = []
while scheme.c.time < config.time_end:
    ens = []
    for m in range(scheme.c.nens):
        forecast = scheme.c.io.call_method(scheme.c, 'current', model.read_var, name='velocity', member=m, time=scheme.c.time, model_src='vort2d')
        ens.append(forecast)
    forecast_state.append(ens)

    scheme.c.time = scheme.c.next_time

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
nt = len(truth_state)

def plot_frame(n):
    Xt = truth_state[n]
    X = forecast_state[n]
    Xb = forecast_state[n]

    # panel 1: true vorticity map, highlighted contour in black, and ensemble members in colors
    ax[0].clear()
    ax[0].pcolor(uv2zeta(Xt), vmin=vort_min, vmax=vort_max, cmap=vort_cmap)
    clvs = [-1e-3, 1e-3]
    cmap = [plt.cm.jet(x) for x in np.linspace(0, 1, scheme.c.nens)]
    for m in range(scheme.c.nens):
        ax[0].contour(uv2zeta(Xb[m]), clvs, colors=[cmap[m][0:3]], linewidths=1)
    ax[0].contour(uv2zeta(Xt), clvs, colors='k', linewidths=2)
    ax[0].set_title(f"Vorticity t={n*config.cycle_period}h")

    # panel 2: time series of error and ensemble spread
    ax[1].clear()
    tt, rmse_st = sawtooth(rmse(Xb, Xt), rmse(X, Xt))
    tt, sprd_st = sawtooth(sprd(Xb), sprd(X))
    print(tt, rmse_st, sprd_st)
    ax[1].plot(tt[0:2*n+1], rmse_st[0:2*n+1], color='red', linewidth=3)
    ax[1].plot(tt[0:2*n+1], sprd_st[0:2*n+1], color='red', linestyle=':', linewidth=2)
    ax[1].set_title('domain-avg rmse,sprd')

    # panel 3: power spectra of error and ensemble spread
    # ax[2].clear()
    # wn, pwr = pwr_spec(Xt[:, :, :, t])
    # ax[2].loglog(wn, pwr, color='black', linewidth=3, label='truth')
    # wn, err_pwr = err_spec(X[:, :, :, :, t], Xt[:, :, :, t])
    # ax[2].loglog(wn, err_pwr, color='red', linewidth=3, label='error')
    # wn, sprd_pwr = sprd_spec(X[:, :, :, :, t])
    # ax[2].loglog(wn, sprd_pwr, color='red', linestyle=':', linewidth=2, label='sprd')
    # ax[2].legend()
    # ax[2].set_title('power spectrum t={}h'.format(t))
    # ax[2].set_ylim([1e-3, 1e2])

plt.close()
ani = FuncAnimation(fig, plot_frame, frames=range(45), interval=300)
HTML(ani.to_jshtml())

## Substeps in an analysis cycle

In [ ]:
scheme.c.time = config.time_start

## Running the entire scheme

Cycling from time_start to time_end

In [ ]:
scheme.c.time = config.time_start
scheme.c.progress.call_stack = []

scheme()


In [ ]:
scheme.c.time = config.time_start
prior_state = []
post_state = []
while scheme.c.time < config.time_end:

    path = scheme.c.fs.forecast_dir(scheme.c.prev_time, 'vort2d')
    ens = []
    for m in range(scheme.c.nens):
        ens.append(model.read_var(path=path, name='velocity', member=m, time=scheme.c.time))
    prior_state.append(ens)

    ens = []
    for m in range(scheme.c.nens):
        if config.run_analysis and scheme.c.time >= config.time_analysis_start and scheme.c.time <= config.time_analysis_end:
            path = scheme.c.fs.forecast_dir(scheme.c.time, 'vort2d')
            ens.append(model.read_var(path=path, name='velocity', member=m, time=scheme.c.time))
        else:
            ens.append(np.full(truth.shape, np.nan))
    post_state.append(ens)

    scheme.c.time = scheme.c.next_time

In [ ]:
from NEDAS.utils.spatial_operation import gradx, grady

def to_vorticity(vel):
    u, v = vel[0], vel[1]
    zeta = gradx(v, grid.dx, grid.cyclic_dim) - grady(u, grid.dy, grid.cyclic_dim)
    return zeta

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
nt = 10 #len(truth_state)
clvs = [1e-3]
current_contour_sets = []

def update(n):
    global current_contour_sets
    for cs in current_contour_sets:
        cs.remove()
    current_contour_sets.clear()

    pc_truth = ax[0].pcolormesh(to_vorticity(truth_state[n]), vmin=-5e-3, vmax=5e-3, cmap='bwr')
    current_contour_sets.append(pc_truth)
    cmap = [plt.cm.jet(x) for x in np.linspace(0, 1, scheme.c.nens)]
    for m in range(scheme.c.nens):
        cs_ens = ax[0].contour(to_vorticity(prior_state[n][m]), clvs, colors=[cmap[m][0:3]], linewidths=0.5)
        current_contour_sets.append(cs_ens)
    cs_truth = ax[0].contour(to_vorticity(truth_state[n]), clvs, colors='k', linewidths=1.5)
    current_contour_sets.append(cs_truth)
    ax[0].set_title(f"Forecast {n*config.cycle_period} h")
    return current_contour_sets

ani = FuncAnimation(fig, update, frames=range(nt), blit=False, interval=100)
plt.close()
HTML(ani.to_jshtml())

## A few things to try to play with
